# Typing the notes

Session 2, *The source and the notes*, part 5.

Besides OMR, the other way to get notes into the computer is to type them. You type the bass (left hand) of the pickup and bar 1 of Schumann's *Fröhlicher Landmann*, as in the first system of page 8:

- pickup: C3, eighth
- bar 1: F3 dotted quarter, A3 eighth, C4 dotted quarter, F3 eighth

The piece is in F major (one flat), in common time, in the bass clef. Middle C is C4.

**How to use this notebook:** run the cells from top to bottom with Shift+Enter. Edit the text between the triple quotes `"""` and run the cell again. Everyone does part 1; part 2 is an extension.

## Setup

The first cell makes sure Verovio is installed; after the setup it already is, so the cell finishes quickly. The second cell defines helper functions; it is collapsed because you do not need to read it.

In [ ]:
%pip install -q verovio==6.3.0

In [1]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

# Read pitches, metre and key from the MEI that Verovio writes.
#
# Why not Verovio's own MIDI values? Verovio 6.3 drops gestural accidentals
# when it imports **kern: with *k[f#c#], the token 4f# becomes
# <note pname="f"/> and getMIDIValuesForElement() reports F natural.
# So we apply the key signature ourselves:
#   accid.ges > accid > accidental earlier in the bar > key signature.

import re

STEPS = {"c": 0, "d": 2, "e": 4, "f": 5, "g": 7, "a": 9, "b": 11}
ACCID = {"s": 1, "f": -1, "ss": 2, "x": 2, "ff": -2, "n": 0, "xs": 3, "ts": 3, "tf": -3}
SHARP_ORDER = "fcgdaeb"
FLAT_ORDER = "beadgcf"


def _attrs(text):
    out = {}
    for m in re.finditer(r'([\w.:]+)="([^"]*)"', text, re.ASCII):
        out[m.group(1)] = m.group(2)
    return out


def _key_from_sig(sig):
    # "2s" -> {f: 1, c: 1}; "0" -> {}
    m = re.match(r"^(\d+)([sf]?)$", sig or "")
    key = {}
    if not m:
        return key
    n = int(m.group(1))
    order = FLAT_ORDER if m.group(2) == "f" else SHARP_ORDER
    for i in range(n):
        key[order[i]] = -1 if m.group(2) == "f" else 1
    return key


# Map xml:id -> MIDI pitch for every <note> with pname and oct.
def pitches_from_mei(mei):
    pitches = {}
    key = {}
    carried = {}  # "staff:pname:oct" -> alteration, reset per measure
    staff = "1"
    note = None  # open <note> waiting for a child <accid>

    def finish(n):
        where = f"{staff}:{n['pname']}:{n['oct']}"
        if n.get("ges") is not None:
            alter = n["ges"]
        elif n.get("written") is not None:
            alter = n["written"]
        elif where in carried:
            alter = carried[where]
        else:
            alter = key.get(n["pname"], 0)
        if n.get("written") is not None:
            carried[where] = n["written"]
        pitches[n["id"]] = 12 * (int(n["oct"]) + 1) + STEPS[n["pname"]] + alter

    for m in re.finditer(r"<(/?)([\w]+)\b([^>]*?)(/?)>", mei, re.ASCII):
        closing, tag, body, self_closing = m.group(1), m.group(2), m.group(3), m.group(4)
        if closing:
            if tag == "note" and note:
                finish(note)
                note = None
            continue
        a = _attrs(body)
        if "keysig" in a:
            key = _key_from_sig(a["keysig"])
        if tag == "keySig" and "sig" in a:
            key = _key_from_sig(a["sig"])
        if tag == "measure":
            carried = {}
        if tag == "staff" and a.get("n"):
            staff = a["n"]
        if tag == "note" and a.get("pname") and "oct" in a:
            n = {
                "id": a.get("xml:id"),
                "pname": a["pname"],
                "oct": a["oct"],
                "ges": ACCID.get(a.get("accid.ges")),
                "written": ACCID.get(a.get("accid")),
            }
            if self_closing:
                finish(n)
            else:
                note = n
        if tag == "accid" and note:
            if "accid.ges" in a:
                note["ges"] = ACCID.get(a["accid.ges"])
            if "accid" in a:
                note["written"] = ACCID.get(a["accid"])
    return pitches


# Metre {count, unit}; 4/4 if the MEI has none.
def meter_from_mei(mei):
    tag = re.search(r"<meterSig\b[^>]*>", mei, re.ASCII)
    source = tag.group(0) if tag else mei
    count = re.search(r'\b(?:meter\.)?count="(\d+)"', source, re.ASCII)
    unit = re.search(r'\b(?:meter\.)?unit="(\d+)"', source, re.ASCII)
    if count and unit:
        return {"count": int(count.group(1)), "unit": int(unit.group(1))}
    return {"count": 4, "unit": 4}


# First key signature as fifths: 2 = two sharps, -3 = three flats.
def fifths_from_mei(mei):
    m = re.search(r'\bkeysig="(\d+)([sf]?)"', mei, re.ASCII) or re.search(
        r'<keySig\b[^>]*\bsig="(\d+)([sf]?)"', mei, re.ASCII
    )
    if not m:
        return 0
    n = int(m.group(1))
    return -n if m.group(2) == "f" else n

# Melody <-> binary pitch x time grid (Demo C). Pure functions, no DOM.
# The Python version in 01_binary_grid.ipynb follows the same rules.
#
# A cell covers [i / res, (i + 1) / res) quarter notes. Two sets of rules:
# "demo": a note fills every cell whose start lies inside the note (what sounds
#   at the start of the cell); notes that cover no cell start are lost. The grid
#   is read back column by column, as a sequence of chords.
# "camat": as CAMAT's create_binary_matrix, onsets are floored and ends ceiled,
#   so a note fills every cell it touches. The grid is read back row by row, as
#   CAMAT's binary_slice_to_df does: each run of 1s is one note. CAMAT has no
#   onset layer (yet), so this read-back ignores onsets.

import math

NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]


def pitch_name(midi):
    return NAMES[midi % 12] + str(math.floor(midi / 12) - 1)


def is_black_key(midi):
    return len(NAMES[midi % 12]) > 1


# Notes from a Verovio timemap: [{pitch, on, off}] in quarter notes.
# pitches: dict xml:id -> MIDI pitch (pitches_from_mei in mei.py); ids not in
# the map (rests) are skipped.
def notes_from_timemap(timemap, pitches):
    offs = {}
    for event in timemap:
        for id_ in event.get("off") or []:
            offs[id_] = event["qstamp"]
    notes = []
    for event in timemap:
        for id_ in event.get("on") or []:
            if id_ not in pitches:
                continue
            notes.append({
                "pitch": pitches[id_],
                "on": event["qstamp"],
                "off": offs.get(id_, event["qstamp"]),
            })
    notes.sort(key=lambda n: (n["on"], n["pitch"]))
    return notes


def bar_quarters(meter):
    return (meter["count"] * 4) / meter["unit"]


# Start of the cell grid: round to avoid 0.9999 artefacts.
EPS = 1e-9


def _js_round(x):
    # Math.round is round-half-up; Python round() is banker's rounding.
    return math.floor(x + 0.5)


# to_grid(notes, res, meter, rule) — positional (JS passes {res, meter, rule}).
def to_grid(notes, res, meter, rule="demo"):
    bar_cells = _js_round(bar_quarters(meter) * res)
    end = max([0] + [n["off"] for n in notes])
    bars = max(1, math.ceil(end / bar_quarters(meter) - EPS))
    n_cols = bars * bar_cells
    pitch_list = [n["pitch"] for n in notes]
    # Math.min(...[]) is Infinity; Math.max(...[]) is -Infinity.
    low = min(pitch_list) if pitch_list else float("inf")
    high = max(pitch_list) if pitch_list else float("-inf")
    occ = {}
    ons = {}
    p = low
    while p <= high:
        occ[p] = [0] * n_cols
        ons[p] = [0] * n_cols
        p += 1
    lost = []
    moved = []  # kept, but starting or ending at a different time
    for n in notes:
        if rule == "camat":
            first = math.floor(n["on"] * res + EPS)
        else:
            first = math.ceil(n["on"] * res - EPS)
        stop = math.ceil(n["off"] * res - EPS)  # first cell not covered
        if first >= stop or n["off"] - n["on"] < EPS:
            lost.append(n)
            continue
        if abs(first - n["on"] * res) > EPS or abs(stop - n["off"] * res) > EPS:
            moved.append(n)
        i = first
        while i < stop and i < n_cols:
            occ[n["pitch"]][i] = 1
            i += 1
        ons[n["pitch"]][first] = 1
    return {
        "res": res,
        "meter": meter,
        "rule": rule,
        "barCells": bar_cells,
        "nCols": n_cols,
        "low": low,
        "high": high,
        "occ": occ,
        "ons": ons,
        "lost": lost,
        "moved": moved,
    }


# Rows to show, highest first. compact: hide black-key rows that stay empty.
def grid_rows(grid, compact=True):
    rows = []
    p = grid["high"]
    while p >= grid["low"]:
        if compact and is_black_key(p) and not any(grid["occ"][p]):
            p -= 1
            continue
        rows.append(p)
        p -= 1
    return rows


# Grid as text, one row per pitch, cells grouped by bar (as on the slides).
def grid_to_text(grid, compact=True, onsets=False):
    rows = grid_rows(grid, compact)
    bars = grid["nCols"] / grid["barCells"]
    width = max(grid["barCells"], 5) + 2

    def label(s):
        return s.ljust(6)

    def block(cells):
        header = label("") + "".join(
            f"bar {b + 1}".ljust(width) for b in range(int(bars))
        ).rstrip()
        lines = [header]
        for p in rows:
            line = label(pitch_name(p))
            for b in range(int(bars)):
                chunk = cells[p][b * grid["barCells"]:(b + 1) * grid["barCells"]]
                line += "".join(str(v) for v in chunk).ljust(width)
            lines.append(line.rstrip())
        return "\n".join(lines)

    if not onsets:
        return block(grid["occ"])
    # Both grids in one: a note can only start where a pitch sounds, so three values are enough.
    three = {p: [(2 if grid["ons"][p][i] else 1) if v else 0 for i, v in enumerate(grid["occ"][p])] for p in rows}
    return (
        f"Sounding (1 = the pitch sounds):\n{block(grid['occ'])}\n\n"
        f"Onsets (1 = a note starts):\n{block(grid['ons'])}\n\n"
        f"Both in one grid (2 = a note starts, 1 = it continues, 0 = silence):\n{block(three)}"
    )


# Read the grid back as events: a new event starts whenever the set of
# sounding pitches changes or (with onsets) a sounding pitch has an onset.
def grid_to_events(grid, onsets=False):
    events = []
    current = None
    for i in range(grid["nCols"]):
        active = []
        attack = False
        p = grid["low"]
        while p <= grid["high"]:
            if grid["occ"][p][i]:
                active.append(p)
                if onsets and grid["ons"][p][i]:
                    attack = True
            p += 1
        key = ",".join(str(x) for x in active)
        if current and current["key"] == key and not attack:
            current["len"] += 1
        else:
            current = {"key": key, "start": i, "len": 1, "pitches": active}
            events.append(current)
    return [{"start": e["start"], "len": e["len"], "pitches": e["pitches"]} for e in events]


# Read the grid back with its rules: {events (for events_to_kern), notes, cut}.
# cut: how often the column-by-column reading cuts a held note because
# another pitch starts or stops while it sounds.
def read_grid(grid, onsets=False):
    if grid.get("rule") == "camat":
        notes = grid_to_notes(grid)
        return {"events": notes_to_events(notes, grid), "notes": notes, "cut": 0}
    events = grid_to_events(grid, onsets)
    cut = 0
    for k in range(1, len(events)):
        i = events[k]["start"]
        for p in events[k]["pitches"]:
            if p in events[k - 1]["pitches"] and not (onsets and grid["ons"][p][i]):
                cut += 1
    return {"events": events, "notes": events_to_notes(events, grid["res"]), "cut": cut}


# Row by row, as CAMAT reads its grids: each run of 1s in one row is one note,
# so repeated notes of the same pitch merge.
def grid_to_notes(grid):
    notes = []
    p = grid["low"]
    while p <= grid["high"]:
        start = -1
        for i in range(grid["nCols"] + 1):
            on = i < grid["nCols"] and grid["occ"][p][i] == 1
            if start >= 0 and not on:
                notes.append({"pitch": p, "on": start / grid["res"], "off": i / grid["res"]})
                start = -1
            if on and start < 0:
                start = i
        p += 1
    notes.sort(key=lambda n: (n["on"], n["pitch"]))
    return notes


# Notes -> events for events_to_kern: cut wherever a note starts or ends.
# tied: pitches that go on sounding from the event before.
def notes_to_events(notes, grid):
    res = grid["res"]
    spans = [(n["pitch"], _js_round(n["on"] * res), _js_round(n["off"] * res)) for n in notes]
    cuts = sorted({0, grid["nCols"], *(s[1] for s in spans), *(s[2] for s in spans)})
    events = []
    for k in range(len(cuts) - 1):
        here = [s for s in spans if s[1] <= cuts[k] < s[2]]
        events.append({
            "start": cuts[k],
            "len": cuts[k + 1] - cuts[k],
            "pitches": sorted(s[0] for s in here),
            "tied": sorted(s[0] for s in here if s[1] < cuts[k]),
        })
    return events


def events_to_notes(events, res):
    notes = []
    for e in events:
        for p in e["pitches"]:
            notes.append({
                "pitch": p,
                "on": e["start"] / res,
                "off": (e["start"] + e["len"]) / res,
            })
    notes.sort(key=lambda n: (n["on"], n["pitch"]))
    return notes


# kern pitch: c = C4, cc = C5, C = C3, CC = C2.
def kern_pitch(midi):
    name = NAMES[midi % 12]
    octave = math.floor(midi / 12) - 1
    letter = name[0]
    accidental = "#" if len(name) > 1 else ""
    repeat = octave - 3 if octave >= 4 else 4 - octave
    base = letter.lower() if octave >= 4 else letter
    return base * repeat + accidental


# Durations kern can write as one token, in quarter notes, largest first.
KERN_DURATIONS = [
    (6, "1."), (4, "1"), (3, "2."), (2, "2"), (1.5, "4."), (1, "4"),
    (0.75, "8."), (0.5, "8"), (0.375, "16."), (0.25, "16"), (0.125, "32"),
]


def _split_duration(quarters):
    parts = []
    left = quarters
    while left > EPS:
        fit = next(((q, recip) for q, recip in KERN_DURATIONS if q <= left + EPS), None)
        if not fit:
            break
        parts.append(fit)
        left -= fit[0]
    return parts


# Events (from grid_to_events or notes_to_events) -> a **kern score Verovio can render.
def events_to_kern(events, grid):
    res, meter, bar_cells = grid["res"], grid["meter"], grid["barCells"]
    sounding = [p for e in events for p in e["pitches"]]
    sorted_p = sorted(sounding)
    median = sorted_p[len(sorted_p) // 2] if sorted_p else 60
    lines = [
        "**kern",
        "*clefF4" if median < 57 else "*clefG2",
        f"*M{meter['count']}/{meter['unit']}",
    ]
    # Cut each event at bar lines, then into writable durations.
    pieces = []
    for k, e in enumerate(events):
        tied_in = e.get("tied", [])
        tied_out = events[k + 1].get("tied", []) if k + 1 < len(events) else []
        start = e["start"]
        end = e["start"] + e["len"]
        chunks = []
        while start < end:
            bar_end = (math.floor(start / bar_cells) + 1) * bar_cells
            stop = min(end, bar_end)
            for q, recip in _split_duration((stop - start) / res):
                chunks.append({"recip": recip, "start": start, "bar": math.floor(start / bar_cells)})
            start = stop
        for j, c in enumerate(chunks):
            pieces.append({**c, "pitches": e["pitches"], "first": j == 0, "last": j == len(chunks) - 1,
                           "tied_in": tied_in, "tied_out": tied_out})
    bar = -1
    for piece in pieces:
        if piece["bar"] != bar:
            bar = piece["bar"]
            lines.append(f"={bar + 1}")
        if not piece["pitches"]:
            lines.append(f"{piece['recip']}r")
            continue
        tokens = []
        for p in piece["pitches"]:
            token = piece["recip"] + kern_pitch(p)
            into = not piece["first"] or p in piece["tied_in"]
            onward = not piece["last"] or p in piece["tied_out"]
            if into and onward:
                tokens.append(token + "_")
            elif onward:
                tokens.append("[" + token)
            elif into:
                tokens.append(token + "]")
            else:
                tokens.append(token)
        lines.append(" ".join(tokens))
    lines.extend(["==", "*-"])
    return "\n".join(lines) + "\n"


# Compare two note lists: same pitches at the same times?
# Key by (pitch, on, off) so 1 and 1.0 match (JS string keys would differ).
def compare_notes(before, after):
    def key(n):
        return (n["pitch"], n["on"], n["off"])

    after_keys = {key(n) for n in after}
    before_keys = {key(n) for n in before}
    return {
        "kept": sum(1 for n in before if key(n) in after_keys),
        "missing": [n for n in before if key(n) not in after_keys],
        "added": [n for n in after if key(n) not in before_keys],
    }

# Input checks before Verovio sees the text, and melody checks for Demo D.
# Pure functions, no DOM. The notebooks carry the same rules in Python.

import re


# Some broken inputs crash Verovio 6.3 instead of giving an error, and some
# valid ones are misread (tested 2026-10-14). errors: do not load the text;
# warnings: load it, but say what Verovio will get wrong.
def precheck(fmt, text):
    problems = []
    warnings = []
    if fmt == "abc":
        lines = re.split(r"\r?\n", text)
        if not any(re.match(r"^X:", l) for l in lines):
            problems.append("ABC needs an X: line (for example X:1) before the tune.")
        if not any(re.match(r"^K:", l) for l in lines):
            problems.append("ABC needs a K: line (the key, for example K:C) before the notes.")
        depth = 0
        unmatched = False
        chord_length = False
        for line in lines:
            if re.match(r"^[A-Za-z]:", line):
                continue  # header and lyrics lines
            body = re.sub(r"%.*$", "", line)
            body = re.sub(r'"[^"]*"', "", body)
            body = re.sub(r"\|\]|\[\|", "|", body)
            if re.search(r"\][\d/]", body):
                chord_length = True
            for ch in body:
                if ch == "[":
                    depth += 1
                if ch == "]":
                    if depth == 0:
                        unmatched = True
                    else:
                        depth -= 1
        if unmatched:
            problems.append("There is a ] without a matching [ in the notes. Use |] for a final bar line.")
        if chord_length:
            warnings.append(
                "Verovio 6.3 ignores a length written after a chord, as in [CEG]2, and plays it as one unit. "
                "Write the length inside: [C2E2G2]."
            )
    if fmt == "humdrum":
        lines = [l for l in re.split(r"\r?\n", text) if l and not l.startswith("!!")]
        head = next((l for l in lines if l.startswith("**")), None)
        if head and re.search(r" {2,}|\*\*\w+ +\*\*", head, re.ASCII) and "\t" not in head:
            problems.append("Humdrum columns (spines) must be separated by tabs, not spaces.")
        manipulators = any(
            any(re.match(r"^\*[\^v+x]$", t) for t in l.split("\t"))
            for l in lines
        )
        if head and not manipulators:
            width = len(head.split("\t"))
            bad = next((i for i, l in enumerate(lines) if len(l.split("\t")) != width), -1)
            if bad >= 0:
                n_cols = len(lines[bad].split("\t"))
                problems.append(
                    f'Line "{lines[bad]}" has {n_cols} column(s), '
                    f"but the file has {width}. Every line needs one entry per spine."
                )
    return {"errors": problems, "warnings": warnings}


# LLM answers often come inside a Markdown code fence: keep only its content.
def clean_answer(text):
    fence = re.search(r"```[\w-]*\n([\s\S]*?)```", text, re.ASCII)
    return ((fence.group(1) if fence else text).strip() + "\n")


# Bar lengths in quarter notes, from the timemap's measure starts.
def bar_lengths(timemap, notes):
    starts = [e["qstamp"] for e in timemap if e.get("measureOn")]
    end = max(
        [0]
        + [e["qstamp"] for e in timemap]
        + [n["off"] for n in notes]
    )
    return [
        (starts[i + 1] if i + 1 < len(starts) else end) - s
        for i, s in enumerate(starts)
    ]


def parse_pitch(name):
    m = re.match(r"^([A-Ga-g])([#b]?)(-?\d)$", name.strip())
    if not m:
        return None
    step = {"c": 0, "d": 2, "e": 4, "f": 5, "g": 7, "a": 9, "b": 11}[m.group(1).lower()]
    alter = 1 if m.group(2) == "#" else -1 if m.group(2) == "b" else 0
    return 12 * (int(m.group(3)) + 1) + step + alter


def _js_num(n):
    # JS template strings write 2 for 2.0 and 3.5 for 3.5.
    if isinstance(n, float) and n.is_integer():
        return str(int(n))
    return str(n)


# Compare what Verovio understood with what the prompt asked for.
# found: {meter, fifths, bars: [lengths], notes}; spec: {meter, fifths, bars, low, high, final}
def melody_checks(found, spec):
    checks = []

    def add(label, ok, detail):
        checks.append({"label": label, "ok": ok, "detail": detail})

    want = (spec["meter"]["count"] * 4) / spec["meter"]["unit"]
    add(
        f"Metre is {spec['meter']['count']}/{spec['meter']['unit']}",
        found["meter"]["count"] == spec["meter"]["count"] and found["meter"]["unit"] == spec["meter"]["unit"],
        f"found {found['meter']['count']}/{found['meter']['unit']}",
    )
    add(f"{spec['bars']} bars", len(found["bars"]) == spec["bars"], f"found {len(found['bars'])}")
    wrong = [[i + 1, q] for i, q in enumerate(found["bars"]) if abs(q - want) > 1e-6]
    add(
        f"Every bar adds up to {_js_num(want)} quarter notes",
        len(wrong) == 0,
        ", ".join(f"bar {b}: {_js_num(q)}" for b, q in wrong) if wrong else "all bars complete",
    )

    def sig(f):
        if f == 0:
            return "no sharps or flats"
        if f > 0:
            return f"{f} sharp{'s' if f > 1 else ''}"
        return f"{-f} flat{'s' if f < -1 else ''}"

    add(f"Key signature: {sig(spec['fifths'])}", found["fifths"] == spec["fifths"], f"found {sig(found['fifths'])}")
    if found["notes"]:
        pitches = [n["pitch"] for n in found["notes"]]
        low = min(pitches)
        high = max(pitches)
        add(
            f"Range within {pitch_name(spec['low'])}–{pitch_name(spec['high'])}",
            low >= spec["low"] and high <= spec["high"],
            f"found {pitch_name(low)}–{pitch_name(high)}",
        )
        last = found["notes"][-1]["pitch"]
        final_name = re.sub(r"-?\d+$", "", pitch_name(spec["final"]))
        add(
            f"Ends on {final_name}",
            last % 12 == spec["final"] % 12,
            f"last note {pitch_name(last)}",
        )
    else:
        add("Has notes", False, "Verovio found no notes")
    return checks

# Verovio in a separate Python process.
#
# Some broken inputs crash Verovio 6.3 instead of giving an error (tested
# 2026-10-14): ABC without X: or K:, kern with too few columns. In a notebook
# a crash kills the kernel, and all your variables are gone. So Verovio runs
# in a child process: if it crashes, only the child dies. The child's error
# output is Verovio's log, which Jupyter would otherwise not show.

import json
import math
import os
import subprocess
import sys

from IPython.display import HTML, display

VEROVIO_VERSION = "6.3.0"

_CHILD = r'''
import json, sys
import verovio
job = json.loads(sys.stdin.read())
tk = verovio.toolkit()
tk.setOptions(job["options"])
out = {"ok": bool(tk.loadData(job["text"])), "version": tk.getVersion()}
if out["ok"]:
    out["svg"] = [tk.renderToSVG(p) for p in range(1, tk.getPageCount() + 1)] if job["svg"] else []
    out["mei"] = tk.getMEI()
    out["timemap"] = tk.renderToTimemap({"includeMeasures": True, "includeRests": True})
sys.stdout.write("\n@@RESULT@@\n" + json.dumps(out))
'''


# Load text in one format and return what the demos need (as on the web page).
# width: target width in pixels; Verovio lays the music out to fit it.
def read_score(fmt, text, width=800, scale=40, header="auto", svg=True):
    checked = precheck(fmt, text)
    result = {"ok": False, "errors": checked["errors"], "warnings": checked["warnings"],
              "log": "", "svg": [], "mei": "", "timemap": [], "notes": []}
    if result["errors"]:
        return result
    options = {
        "inputFrom": fmt, "scale": scale, "pageWidth": max(600, math.floor(width * 100 / scale + 0.5)),
        "adjustPageHeight": True, "pageMarginTop": 30, "pageMarginBottom": 30,
        "pageMarginLeft": 30, "pageMarginRight": 30, "header": header, "footer": "none",
    }
    child = subprocess.run(
        [sys.executable, "-c", _CHILD],
        input=json.dumps({"text": text, "options": options, "svg": svg}),
        capture_output=True, text=True, encoding="utf-8", timeout=300,
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    result["log"] = child.stderr.strip()
    if child.returncode < 0:
        result["errors"].append(
            f"Verovio crashed on this text (signal {-child.returncode}). This happens with some broken "
            "input that Verovio does not report as an error. Undo your last change and try again.")
        return result
    if child.returncode > 0 or "\n@@RESULT@@\n" not in child.stdout:
        result["errors"].append("The Verovio process failed; see the log below. Did the install cell run?")
        return result
    out = json.loads(child.stdout.rsplit("\n@@RESULT@@\n", 1)[1])
    if not out["ok"]:
        result["errors"].append(f"Verovio could not read this text as {fmt}.")
        return result
    result.update(ok=True, version=out["version"], svg=out["svg"], mei=out["mei"], timemap=out["timemap"])
    result["notes"] = notes_from_timemap(result["timemap"], pitches_from_mei(result["mei"]))
    result["meter"] = meter_from_mei(result["mei"])
    result["fifths"] = fifths_from_mei(result["mei"])
    return result


# The score on a white background (readable in dark themes), then the messages.
def show(result):
    for page in result["svg"]:
        display(HTML(f'<div style="background:white; padding:4px; overflow-x:auto">{page}</div>'))
    for text in result["errors"]:
        print("Error:", text)
    for text in result["warnings"]:
        print("Warning:", text)
    for line in result["log"].splitlines():
        if line.strip():
            print("Verovio:", line)


def render(fmt, text, **options):
    result = read_score(fmt, text, **options)
    show(result)
    return result

# The notes per bar in words, as under the course page's ABC box (demos.js,
# barsRead), and a check against the bass of the pickup and bar 1.

LENGTH_NAMES = {0.25: "sixteenth", 0.5: "eighth", 0.75: "dotted eighth", 1: "quarter",
                1.5: "dotted quarter", 2: "half", 3: "dotted half", 4: "whole"}
SHARP_NAMES = ["C", "C♯", "D", "D♯", "E", "F", "F♯", "G", "G♯", "A", "A♯", "B"]
FLAT_NAMES = ["C", "D♭", "D", "E♭", "E", "F", "G♭", "G", "A♭", "A", "B♭", "B"]

# The print: pickup C3 eighth; bar 1 F3 dotted quarter, A3 eighth, C4 dotted quarter, F3 eighth.
BASS_PRINT = [("Pickup", ["C3 eighth"]),
              ("Bar 1", ["F3 dotted quarter", "A3 eighth", "C4 dotted quarter", "F3 eighth"])]


def _number(q):
    return str(int(q)) if q == int(q) else str(q)


def length_name(q):
    return LENGTH_NAMES.get(q, f"{_number(q)} quarter notes")


def spelled(midi, fifths):
    return (FLAT_NAMES if fifths < 0 else SHARP_NAMES)[midi % 12] + str(midi // 12 - 1)


# A first bar shorter than the metre is the pickup; the next one is bar 1.
def bars_read(result):
    starts = [e["qstamp"] for e in result["timemap"] if e.get("measureOn")]
    lengths = bar_lengths(result["timemap"], result["notes"])
    full = result["meter"]["count"] * 4 / result["meter"]["unit"]
    pickup = bool(lengths) and lengths[0] < full - 1e-6
    events = [(n["on"], f"{spelled(n['pitch'], result['fifths'])} {length_name(n['off'] - n['on'])}")
              for n in result["notes"]]
    rest_ends = {i: e["qstamp"] for e in result["timemap"] for i in e.get("restsOff") or []}
    events += [(e["qstamp"], f"rest {length_name(rest_ends.get(i, e['qstamp']) - e['qstamp'])}")
               for e in result["timemap"] for i in e.get("restsOn") or []]
    events.sort(key=lambda event: event[0])
    bars = []
    for i, start in enumerate(starts):
        end = starts[i + 1] if i + 1 < len(starts) else float("inf")
        is_pickup = pickup and i == 0
        notes = [text for on, text in events if start - 1e-6 <= on < end - 1e-6]
        short = ("" if is_pickup or abs(lengths[i] - full) < 1e-6
                 else f"{_number(lengths[i])} of {_number(full)} quarter notes")
        bars.append((("Pickup" if is_pickup else f"Bar {i if pickup else i + 1}"), notes, short))
    return bars


def print_bars(result):
    if not result["ok"]:
        return
    print("Verovio reads")
    for label, notes, short in bars_read(result):
        print(f"  {label}: {', '.join(notes) or 'nothing'}" + (f" ({short})" if short else ""))


def check_bass(name, result):
    if not result["ok"]:
        print(f"{name}: Verovio could not read it; see the message under the score.")
        return
    found = {label: (notes, short) for label, notes, short in bars_read(result)}
    expected = dict(BASS_PRINT)
    problems = [f"{label} is missing" for label in expected if label not in found]
    if name == "ABC" and "Bar 1" not in found:
        # Verovio leaves out a last bar that no bar line closes.
        problems.append("type it after the bar line and end it with |]")
    for label, (notes, short) in found.items():
        if label not in expected:
            problems.append(f"{label} is extra")
        elif notes != expected[label]:
            problems.append(f"{label} reads {', '.join(notes) or 'nothing'}" + (f" ({short})" if short else ""))
    if result["fifths"] != -1:
        problems.append("the key is not F major (one flat)")
    if problems:
        print(f"{name}: not yet. " + "; ".join(problems))
        return
    print(f"{name}: ✓ the same notes as the print, in F major")

## 1. Type bar 1 in ABC

ABC is the format from session 1. The text below has the header and the pickup already. Type bar 1 after the bar line `|`, end with `|]`, and run the cell. Three things you need:

- `L:1/8` makes the eighth note the unit: `G,` is an eighth, `G,2` a quarter, `G,3` a dotted quarter.
- A comma puts a note one octave lower: `C` is C4 (middle C), `C,` is C3.
- `|` is a bar line; `|]` ends the piece.

Under the score, *Verovio reads* lists the notes in words. Compare them with the list above, and the score with the print: pitches and octaves, rhythm, pickup, clef and key. The same box is on the course page, part 5.

In [2]:
abc = """X:1
T:Fröhlicher Landmann, left hand
M:C
L:1/8
K:F clef=bass
% The pickup is given. Type bar 1 after the bar line and end with |]
C, |
"""
abc_result = render("abc", abc)
print_bars(abc_result)

Verovio reads
  Pickup: C3 eighth


Then let the computer compare your notes with the print:

In [3]:
check_bass("ABC", abc_result)

ABC: not yet. Bar 1 is missing; type it after the bar line and end it with |]


<details><summary>Solution</summary>

```
C, | F,3 A, C3 F, |]
```

</details>

Now think about the whole piece: 21 bars on two staves, with chords in the right hand. What does an existing encoding save you, and what does it not?

## 2. Extension: the same notes in other formats

The same five notes in three more text formats, and in a notation program. Run each cell and compare: every format should give the same notes. At the end, one cell checks them all.

### Plaine & Easie

A short format for melodies, used by library catalogues for incipits (the first notes of a piece). `@clef:F-4` is the bass clef, `@keysig:bB` one flat, `@timesig:4/4` common time. In the notes, `,` sets octave 3 and `'` octave 4; `8` is an eighth, `4.` a dotted quarter. `/` is a bar line.

In [4]:
pae = """@clef:F-4
@keysig:bB
@timesig:4/4
@data:,8C/4.F8A'4.C,8F//
"""
pae_result = render("pae", pae)
print_bars(pae_result)

Verovio reads
  Pickup: C3 eighth
  Bar 1: F3 dotted quarter, A3 eighth, C4 dotted quarter, F3 eighth


### MEI, written by hand

MEI spells out each note with attributes: `pname` (pitch name), `oct` (octave), `dur` (8 = eighth, 4 = quarter) and `dots`. The pickup bar has `metcon="false"`: it is shorter than the time signature says. Each `xml:id` names one note or bar; these belong to this small exercise only. This is the format of our edition. You can also paste the text into [mei-friend](https://mei-friend.mdw.ac.at/) and edit it there.

In [5]:
mei = """<?xml version="1.0" encoding="UTF-8"?>
<mei xmlns="http://www.music-encoding.org/ns/mei" meiversion="5.1">
  <meiHead>
    <fileDesc>
      <titleStmt>
        <title>Fröhlicher Landmann, left hand</title>
        <composer>Robert Schumann</composer>
      </titleStmt>
      <pubStmt/>
    </fileDesc>
  </meiHead>
  <music><body><mdiv><score>
    <scoreDef meter.count="4" meter.unit="4" keysig="1f">
      <staffGrp><staffDef n="1" lines="5" clef.shape="F" clef.line="4"/></staffGrp>
    </scoreDef>
    <section>
      <measure n="0" xml:id="entry-pickup" metcon="false">
        <staff n="1"><layer n="1">
          <note xml:id="entry-c3" pname="c" oct="3" dur="8"/>
        </layer></staff>
      </measure>
      <measure n="1" xml:id="entry-bar1">
        <staff n="1"><layer n="1">
          <note xml:id="entry-f3a" pname="f" oct="3" dur="4" dots="1"/>
          <note xml:id="entry-a3" pname="a" oct="3" dur="8"/>
          <note xml:id="entry-c4" pname="c" oct="4" dur="4" dots="1"/>
          <note xml:id="entry-f3b" pname="f" oct="3" dur="8"/>
        </layer></staff>
      </measure>
    </section>
  </score></mdiv></body></music>
</mei>
"""
mei_result = render("mei", mei)
print_bars(mei_result)

Verovio reads
  Pickup: C3 eighth
  Bar 1: F3 dotted quarter, A3 eighth, C4 dotted quarter, F3 eighth


### Humdrum `**kern`

The format for music analysis from session 1. One line per note: `8C` is C3, an eighth; `4.F` is F3, a dotted quarter; `4.c` is C4. `=1` is the bar line before bar 1.

In [6]:
kern = """!!!OTL: Fröhlicher Landmann, left hand
**kern
*clefF4
*k[b-]
*M4/4
8C
=1
4.F
8A
4.c
8F
==
*-
"""
kern_result = render("humdrum", kern)
print_bars(kern_result)

Verovio reads
  Pickup: C3 eighth
  Bar 1: F3 dotted quarter, A3 eighth, C4 dotted quarter, F3 eighth


### Compare them all

In [7]:
for name, result in {"ABC": abc_result, "Plaine & Easie": pae_result,
                     "MEI": mei_result, "kern": kern_result}.items():
    check_bass(name, result)

ABC: not yet. Bar 1 is missing; type it after the bar line and end it with |]
Plaine & Easie: ✓ the same notes as the print, in F major
MEI: ✓ the same notes as the print, in F major
kern: ✓ the same notes as the print, in F major


### MuseScore

If you have [MuseScore](https://musescore.org) 4.2 or newer: create a score for one bass-clef staff in F major, 4/4, with an eighth-note pickup ([how to make a pickup bar](https://handbook.musescore.org/notation/rhythm-meter-and-measures/pickup-and-non-metered-measures)). Enter the five notes. Then choose **File → Export → MEI** ([exporting](https://handbook.musescore.org/file-management/file-export)). Open the exported file in [mei-friend](https://mei-friend.mdw.ac.at/), or paste it into the MEI cell above, and check pitches, rhythm, clef, key and pickup.

A notation program is a good way to enter many notes. Once an edition has links to the scan and editorial notes, though, we edit its MEI directly: a round trip through a notation program can lose them.

### Keep your text (optional)

Set `SAVE = True` to save the text of one format, and the MEI Verovio made of it, into a new folder next to this notebook. It never overwrites a file: for a second try, choose another folder name.

In [8]:
from pathlib import Path

SAVE = False
CHOSEN = "abc"  # "abc", "pae", "mei" or "kern"
FOLDER = Path.cwd() / "schumann_note_entry_work/attempt-01"

if SAVE:
    text, result = {"abc": (abc, abc_result), "pae": (pae, pae_result),
                    "mei": (mei, mei_result), "kern": (kern, kern_result)}[CHOSEN]
    if not result["ok"]:
        raise ValueError("Verovio could not read this text; fix it before saving.")
    FOLDER.mkdir(parents=True, exist_ok=False)
    extension = {"kern": "krn"}.get(CHOSEN, CHOSEN)
    (FOLDER / f"input.{extension}").write_text(text, encoding="utf-8")
    (FOLDER / "verovio.mei").write_text(result["mei"], encoding="utf-8")
    print("Saved in", FOLDER)
else:
    print("Off. Set SAVE = True to keep your text.")

Off. Set SAVE = True to keep your text.


### Other ways in

- **mei-friend** imports MusicXML, Humdrum, ABC and Plaine & Easie (menu *File*) and shows the result as MEI.
- **A MIDI keyboard** can enter notes into a notation program, one at a time.
- **A language model** can draft ABC or MEI. Check its result as in session 1: the notes, their octaves and lengths, and the pickup.

More about the formats: [Verovio's input formats](https://book.verovio.org/toolkit-reference/input-formats.html), [MEI in MuseScore](https://music-encoding.org/musescore-doc/).